# Assignment 7

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Load and Preprocess Data


In [2]:
# Load the suicide rates dataset
suicide_data = pd.read_csv('../Datasets/suicide-rates.csv')
print(f"Dataset shape: {suicide_data.shape}")
suicide_data.head()


Dataset shape: (27820, 12)


,country,year,sex,age,suicides_no,population,suicides/100k pop,country-year,HDI for year,gdp_for_year ($),gdp_per_capita ($),generation
0,Albania,1987,male,15-24 years,21,312900,6.71,Albania1987,NaN,"2,156,624,900",796,Generation X
1,Albania,1987,male,35-54 years,16,308000,5.19,Albania1987,NaN,"2,156,624,900",796,Silent
2,Albania,1987,female,15-24 years,14,289700,4.83,Albania1987,NaN,"2,156,624,900",796,Generation X
3,Albania,1987,male,75+ years,1,21800,4.59,Albania1987,NaN,"2,156,624,900",796,G.I. Generation
4,Albania,1987,male,25-34 years,9,274300,3.28,Albania1987,NaN,"2,156,624,900",796,Boomers


In [3]:
# Check for missing values
print("Missing values per column:")
print(suicide_data.isnull().sum())
print("\nDataset info:")
suicide_data.info()


Missing values per column:
country                   0
year                      0
sex                       0
age                       0
suicides_no               0
population                0
suicides/100k pop         0
country-year              0
HDI for year          19456
 gdp_for_year ($)         0
gdp_per_capita ($)        0
generation                0
dtype: int64

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27820 entries, 0 to 27819
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             27820 non-null  object 
 1   year                27820 non-null  int64  
 2   sex                 27820 non-null  object 
 3   age                 27820 non-null  object 
 4   suicides_no         27820 non-null  int64  
 5   population          27820 non-null  int64  
 6   suicides/100k pop   27820 non-null  float64
 7   country-year        27820 non-null  object 
 8   HDI f

In [4]:
# Fill missing HDI values with median 
suicide_data['HDI for year'] = suicide_data['HDI for year'].fillna(suicide_data['HDI for year'].median())

# Drop unnecessary columns for regression
columns_to_drop = ['suicides_no', 'country-year', 'year', ' gdp_for_year ($) ']
suicide_data_clean = suicide_data.drop(columns=columns_to_drop)

print(f"Cleaned dataset shape: {suicide_data_clean.shape}")
suicide_data_clean.head()


Cleaned dataset shape: (27820, 8)


,country,sex,age,population,suicides/100k pop,HDI for year,gdp_per_capita ($),generation
0,Albania,male,15-24 years,312900,6.71,0.779,796,Generation X
1,Albania,male,35-54 years,308000,5.19,0.779,796,Silent
2,Albania,female,15-24 years,289700,4.83,0.779,796,Generation X
3,Albania,male,75+ years,21800,4.59,0.779,796,G.I. Generation
4,Albania,male,25-34 years,274300,3.28,0.779,796,Boomers


## Part 1: One-Hot Encoding and Multiple Linear Regression

We need to one-hot encode categorical variables (country, sex, age, generation) to prepare for linear regression.


In [5]:
# encode country, sex, age, generation
suicide_data_encoded = pd.get_dummies(suicide_data_clean, 
                                       columns=['country', 'sex', 'age', 'generation'],
                                       drop_first=False)

print(f"Encoded dataset shape: {suicide_data_encoded.shape}")
print(f"\nColumn names after encoding:")
print(list(suicide_data_encoded.columns))


Encoded dataset shape: (27820, 119)

Column names after encoding:
['population', 'suicides/100k pop', 'HDI for year', 'gdp_per_capita ($)', 'country_Albania', 'country_Antigua and Barbuda', 'country_Argentina', 'country_Armenia', 'country_Aruba', 'country_Australia', 'country_Austria', 'country_Azerbaijan', 'country_Bahamas', 'country_Bahrain', 'country_Barbados', 'country_Belarus', 'country_Belgium', 'country_Belize', 'country_Bosnia and Herzegovina', 'country_Brazil', 'country_Bulgaria', 'country_Cabo Verde', 'country_Canada', 'country_Chile', 'country_Colombia', 'country_Costa Rica', 'country_Croatia', 'country_Cuba', 'country_Cyprus', 'country_Czech Republic', 'country_Denmark', 'country_Dominica', 'country_Ecuador', 'country_El Salvador', 'country_Estonia', 'country_Fiji', 'country_Finland', 'country_France', 'country_Georgia', 'country_Germany', 'country_Greece', 'country_Grenada', 'country_Guatemala', 'country_Guyana', 'country_Hungary', 'country_Iceland', 'country_Ireland', 'co

In [6]:
X = suicide_data_encoded.drop(columns=['suicides/100k pop'])
y = suicide_data_encoded['suicides/100k pop']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nNumber of features: {X.shape[1]}")


Features shape: (27820, 118)
Target shape: (27820,)

Number of features: 118


In [7]:
lr_model = LinearRegression()
lr_model.fit(X, y)


print(f"Number of regression coefficients: {len(lr_model.coef_)}")
print(f"Intercept: {lr_model.intercept_:.4f}")

print("\nRegression coefficients:")
for feature, coef in zip(X.columns, lr_model.coef_):
    print(f"{feature}: {coef:.4f}")


Number of regression coefficients: 118
Intercept: 17.3425

Regression coefficients:
population: 0.0000
HDI for year: -4.6712
gdp_per_capita ($): -0.0001
country_Albania: -9.8807
country_Antigua and Barbuda: -11.8745
country_Argentina: -2.5437
country_Armenia: -10.1506
country_Aruba: -1.2766
country_Australia: 2.8162
country_Austria: 13.7385
country_Azerbaijan: -11.6727
country_Bahamas: -9.5114
country_Bahrain: -9.6412
country_Barbados: -9.3113
country_Belarus: 17.8911
country_Belgium: 11.0444
country_Belize: -6.9167
country_Bosnia and Herzegovina: -8.3198
country_Brazil: -8.7446
country_Bulgaria: 6.2379
country_Cabo Verde: -1.6116
country_Canada: 1.9668
country_Chile: -2.3208
country_Colombia: -8.2274
country_Costa Rica: -5.9979
country_Croatia: 10.5046
country_Cuba: 8.1797
country_Cyprus: -7.1260
country_Czech Republic: 6.3146
country_Denmark: 5.9689
country_Dominica: -13.9268
country_Ecuador: -7.0788
country_El Salvador: -2.8768
country_Estonia: 15.2249
country_Fiji: -7.7419
country_

## Part 2: Prediction for Age 20, Male, Generation X

In [8]:
# check what categories exist in the dataset
print("age", suicide_data['age'].unique())
print("generation", suicide_data['generation'].unique())
print("sex", suicide_data['sex'].unique())


age ['15-24 years' '35-54 years' '75+ years' '25-34 years' '55-74 years'
 '5-14 years']
generation ['Generation X' 'Silent' 'G.I. Generation' 'Boomers' 'Millenials'
 'Generation Z']
sex ['male' 'female']


In [9]:
# Filter the encoded dataset: age "15-24 years", sex "male", generation "generation x"
mask = (suicide_data_clean['age'] == '15-24 years') & (suicide_data_clean['sex'] == 'male') & (suicide_data_clean['generation'] == 'Generation X')

subset_data = suicide_data_clean[mask]
print(f"Number of samples matching our criteria: {len(subset_data)}")
print(subset_data.head())


Number of samples matching our criteria: 1057
    country   sex          age  population  suicides/100k pop  HDI for year  \
0   Albania  male  15-24 years      312900               6.71         0.779   
13  Albania  male  15-24 years      319200               5.33         0.779   
28  Albania  male  15-24 years      323500               3.71         0.779   
37  Albania  male  15-24 years      263700               3.41         0.779   
48  Albania  male  15-24 years      243300               7.40         0.779   

    gdp_per_capita ($)    generation  
0                  796  Generation X  
13                 769  Generation X  
28                 833  Generation X  
37                 251  Generation X  
48                 437  Generation X  


In [10]:
# Get the subset of data that matches our criteria
subset_encoded = suicide_data_encoded.loc[subset_data.index]

X_subset = subset_encoded.drop(columns=['suicides/100k pop'])
y_subset = subset_encoded['suicides/100k pop']

# Make predictions 
y_pred_subset = lr_model.predict(X_subset)

# Calculate MAE
mae_subset = mean_absolute_error(y_subset, y_pred_subset)

print(f"Predictions for age 20 (15-24 years), male, Generation X:")
print(f"Number of predictions: {len(y_pred_subset)}")
print(f"Mean prediction: {y_pred_subset.mean():.4f}")
print(f"Mean actual value: {y_subset.mean():.4f}")
print(f"\nMAE error for these predictions: {mae_subset:.4f}")


Predictions for age 20 (15-24 years), male, Generation X:
Number of predictions: 1057
Mean prediction: 17.1838
Mean actual value: 14.5130

MAE error for these predictions: 6.7805


## Model Performance 


In [11]:
def report_model_performance(X, y):
    y_pred_all = lr_model.predict(X)

    mae_all = mean_absolute_error(y, y_pred_all)
    mse_all = mean_squared_error(y, y_pred_all)
    rmse_all = np.sqrt(mse_all)
    r2_all = r2_score(y, y_pred_all)

    print("Overall Model Performance:")
    print(f"MAE:  {mae_all:.4f}")
    print(f"MSE:  {mse_all:.4f}")
    print(f"RMSE: {rmse_all:.4f}")
    print(f"R^2:   {r2_all:.4f}")


In [12]:
report_model_performance(X, y)

Overall Model Performance:
MAE:  8.6259
MSE:  172.6697
RMSE: 13.1404
R^2:   0.5197


## Part 3: Feature Engineering with Numerical Conversion


In [13]:
# Convert categorical variables to numerical
suicide_data_numeric = suicide_data_clean.copy()

suicide_data_numeric['sex'] = suicide_data_numeric['sex'].map({'male': 1, 'female': 0})

# use midpoint of age
age_mapping = {
    '5-14 years': 9.5,
    '15-24 years': 19.5,
    '25-34 years': 29.5,
    '35-54 years': 44.5,
    '55-74 years': 64.5,
    '75+ years': 80
}
suicide_data_numeric['age'] = suicide_data_numeric['age'].map(age_mapping)

generation_mapping = {
    'G.I. Generation': 1,
    'Silent': 2,
    'Boomers': 3,
    'Generation X': 4,
    'Millenials': 5,
    'Generation Z': 6
}
suicide_data_numeric['generation'] = suicide_data_numeric['generation'].map(generation_mapping)

print(f"Numeric dataset shape: {suicide_data_numeric.shape}")
print(suicide_data_numeric.head())
print(f"\nData types:")
print(suicide_data_numeric.dtypes)


Numeric dataset shape: (27820, 8)
   country  sex   age  population  suicides/100k pop  HDI for year  \
0  Albania    1  19.5      312900               6.71         0.779   
1  Albania    1  44.5      308000               5.19         0.779   
2  Albania    0  19.5      289700               4.83         0.779   
3  Albania    1  80.0       21800               4.59         0.779   
4  Albania    1  29.5      274300               3.28         0.779   

   gdp_per_capita ($)  generation  
0                 796           4  
1                 796           2  
2                 796           4  
3                 796           1  
4                 796           3  

Data types:
country                object
sex                     int64
age                   float64
population              int64
suicides/100k pop     float64
HDI for year          float64
gdp_per_capita ($)      int64
generation              int64
dtype: object


In [14]:
X_numeric = suicide_data_numeric.drop(columns=['suicides/100k pop', 'country'])
y_numeric = suicide_data_numeric['suicides/100k pop']

lr_model_numeric = LinearRegression()
lr_model_numeric.fit(X_numeric, y_numeric)

num_coefficients = len(lr_model_numeric.coef_)
print(f"Number of line coefficients: {num_coefficients}")
print(f"\nFeature names: {list(X_numeric.columns)}")
print(f"\nCoefficients:")
for feature, coef in zip(X_numeric.columns, lr_model_numeric.coef_):
    print(f"{feature}: {coef:.4f}")
print(f"\nIntercept: {lr_model_numeric.intercept_:.4f}")


Number of line coefficients: 6

Feature names: ['sex', 'age', 'population', 'HDI for year', 'gdp_per_capita ($)', 'generation']

Coefficients:
sex: 14.8622
age: 0.2082
population: 0.0000
HDI for year: 18.9466
gdp_per_capita ($): -0.0000
generation: -1.2355

Intercept: -13.8303


## Part 4: Prediction for Age 20, Male, Generation X


In [15]:
mask_numeric = (suicide_data_numeric['age'] == 19.5) & (suicide_data_numeric['sex'] == 1) & (suicide_data_numeric['generation'] == 4)

subset_numeric = suicide_data_numeric[mask_numeric]
print(f"Number of samples matching criteria: {len(subset_numeric)}")

X_subset_numeric = subset_numeric.drop(columns=['suicides/100k pop', 'country'])
y_subset_numeric = subset_numeric['suicides/100k pop']

y_pred_numeric = lr_model_numeric.predict(X_subset_numeric)

mae_numeric = mean_absolute_error(y_subset_numeric, y_pred_numeric)

print(f"\nPredictions for age 20, male, Generation X:")
print(f"Mean prediction: {y_pred_numeric.mean():.4f}")
print(f"Mean actual value: {y_subset_numeric.mean():.4f}")
print(f"\nMAE error: {mae_numeric:.4f}")


Number of samples matching criteria: 1057

Predictions for age 20, male, Generation X:
Mean prediction: 14.8910
Mean actual value: 14.5130

MAE error: 8.8289


Yes, there are some notable differences in performance between the two models. The first model using one-hot encoding has 118 coefficients and was able to achieve an MAE of 6.7805 for the age 20, male, Generation X predictions, with a mean prediction of 17.1838 compared to the actual mean of 14.5130. The second model using numerical conversion has only 6 coefficients and achieved a higher MAE of 8.8289 for the same predictions, but its mean prediction of 14.8910 is much closer to the actual mean.
The one-hot encoded model demonstrates better overall accuracy with its lower MAE, which can be attributed to its ability to capture country-specific patterns through one-hot encoding. However, this comes at the cost of significantly higher model complexity and a prediction that missed the actual mean by about 2.67. The numerical model, while producing a higher error overall, is considerably simpler and more interpretable, showing that being male adds approximately 14.86 to the suicide rate, each year of age adds about 0.21, and each generation step reduces the rate by 1.24.
This is a bias-variance tradeoff where the one-hot encoded model may be overfitting to country-specific patterns while the numerical model provides a more generalizable but slightly less accurate solution. The numerical model's prediction being only 0.38 points higher than the actual mean suggests it has less bias despite having higher variance in its individual predictions.

In [16]:
# create our new data
# age 33, male, and generation Alpha 
# For other features we'll use median values from the dataset 
new_data = pd.DataFrame({
    'sex': [1],
    'age': [33],
    'population': [suicide_data_numeric['population'].median()],
    'HDI for year': [suicide_data_numeric['HDI for year'].median()],
    'gdp_per_capita ($)': [suicide_data_numeric['gdp_per_capita ($)'].median()],
    'generation': [7]  # Gen Z is 6, so Alpha is 7 in this case
})

# Make the prediction using the Q3 numerical model
prediction = lr_model_numeric.predict(new_data)

print(f"Prediction for age 33, male, Generation Alpha:")
print(f"Predicted suicide rate: {prediction[0]:.4f} per 100k population")
print(f"\nInput features used:")
print(new_data)


Prediction for age 33, male, Generation Alpha:
Predicted suicide rate: 13.9368 per 100k population

Input features used:
   sex  age  population  HDI for year  gdp_per_capita ($)  generation
0    1   33    430150.0         0.779              9372.0           7


# 7. Advantages of Regression

One advantage of using regression is that it lets you use numerical independent variables directly. So we don't need to encode categories and increase the dimensionality the feature space. In contrast, classification with nominal features typically requires one-hot encoding, which increases the number of features and makes the model more complex. This also improves interpretability: in our assignment the numerical regression used only 6 coefficients, whereas the one-hot approach produced 118 coefficients that are harder to interpret.

# 8. Advantages of using Numerical Features

Using regular numerical values avoids the one-hot encoding dimensionality increase, keeping the model low-dimensional and simpler to interpret (in our case ~6 coefficients instead of ~118), which makes training faster and reduces overfitting risk.

# 9. Which model I suggest

I’d recommend regression. Our target variable here is continuous, and regression preserves that information, produces estimates, gives clear error metrics (MAE/RMSE), and interpretable effects of predictors. A classifier would require arbitrary bins or thresholds, lose nuance, and be more sensitive to class imbalance. If our customer ultimately needs a yes/no from the model, we can still apply a threshold to the regression output to make that decision.